# 03 — Stdlib (json/csv/pathlib/regex/logging) and Concurrency (asyncio/threading/multiprocessing/GIL)

**Overview:** The stdlib tools every data pipeline uses, plus a hands-on comparison of
concurrency models and a live demonstration of why the GIL makes `threading` useless for
CPU-bound work.

**Required libraries:** standard library only (`json`, `csv`, `pathlib`, `re`, `logging`,
`asyncio`, `threading`, `multiprocessing`, `time`).

**Note:** this notebook spawns worker processes, which behaves differently under Jupyter/Colab
vs a plain script. We keep the multiprocessing demo small and self-contained so it works in
both environments.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **GIL** | Global Interpreter Lock | CPython lock letting only one thread run Python bytecode at a time; limits CPU-bound threading. |
| **CPU** | Central Processing Unit | General-purpose processor; few powerful cores. |
| **JSON** | JavaScript Object Notation | Text format for nested key-value data. |
| **CSV** | Comma-Separated Values | Plain-text table format, one row per line. |
| **I/O** | Input/Output | Reading/writing files, network or devices; usually the slow part that threads/asyncio overlap. |

## 0. Environment

In [1]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())


Python: 3.12.5 (main, Aug 29 2025, 10:17:29) [Clang 17.0.0 (clang-1700.0.13.5)]
Platform: macOS-26.5.1-arm64-arm-64bit


## 1. JSON and CSV

In [2]:
import json, csv, io

records = [{"text": "great movie", "label": 1}, {"text": "terrible plot", "label": 0}]

json_str = json.dumps(records, indent=2)
print(json_str)
parsed_back = json.loads(json_str)
assert parsed_back == records

csv_buffer = io.StringIO()
writer = csv.DictWriter(csv_buffer, fieldnames=["text", "label"])
writer.writeheader()
writer.writerows(records)
print(csv_buffer.getvalue())

csv_buffer.seek(0)
reader = csv.DictReader(csv_buffer)
rows = list(reader)
print("read back:", rows)
assert rows[0]["text"] == "great movie"


[
  {
    "text": "great movie",
    "label": 1
  },
  {
    "text": "terrible plot",
    "label": 0
  }
]
text,label
great movie,1
terrible plot,0

read back: [{'text': 'great movie', 'label': '1'}, {'text': 'terrible plot', 'label': '0'}]


## 2. pathlib

In [3]:
from pathlib import Path

data_dir = Path("scratch_pathlib_demo")
data_dir.mkdir(exist_ok=True)
sample_file = data_dir / "example.txt"
sample_file.write_text("hello from pathlib\n")

print("exists:", sample_file.exists())
print("suffix:", sample_file.suffix, "stem:", sample_file.stem)
print("contents:", sample_file.read_text().strip())
print("all .txt files:", list(data_dir.glob("*.txt")))

sample_file.unlink()
data_dir.rmdir()
assert not sample_file.exists()


exists: True
suffix: .txt stem: example
contents: hello from pathlib
all .txt files: [PosixPath('scratch_pathlib_demo/example.txt')]


## 3. Regular expressions

In [4]:
import re

log_line = "2026-01-15 10:23:45 ERROR Failed to load batch 42: shape mismatch"
pattern = re.compile(r"(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) (?P<level>\w+) (?P<msg>.*)")

m = pattern.match(log_line)
print(m.groupdict())

assert m.group("level") == "ERROR"
assert "shape mismatch" in m.group("msg")

all_numbers = re.findall(r"\d+", log_line)
print("all numbers found:", all_numbers)


{'date': '2026-01-15', 'time': '10:23:45', 'level': 'ERROR', 'msg': 'Failed to load batch 42: shape mismatch'}
all numbers found: ['2026', '01', '15', '10', '23', '45', '42']


## 4. logging

⚠️ **Common Mistake:** using `print()` for diagnostics in library code instead of `logging`, which breaks caller control over verbosity.

In [5]:
import logging

logger = logging.getLogger("ai_ml_fundamentals.demo")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
    logger.addHandler(handler)

logger.info("training started, batch_size=%d", 32)
logger.warning("learning rate %.4f looks unusually high", 1.5)


INFO:ai_ml_fundamentals.demo:training started, batch_size=32


## 5. Concurrency: asyncio vs threading vs multiprocessing, and the GIL

🧠 **Remember:** the GIL lets only one thread execute Python bytecode at a time. Threads still
help for **I/O-bound** work (the GIL is released during I/O waits), but not for **CPU-bound**
work.

🔬 **Experiment 1 — I/O-bound: asyncio beats sequential.**

Jupyter notebooks already run inside an event loop, so we use top-level `await` directly
rather than `asyncio.run()` (which only works from plain scripts, outside any running loop).

In [6]:
import asyncio, time

async def fake_io_call(i: int, delay: float = 0.05):
    await asyncio.sleep(delay)   # simulates a network call
    return i * i

async def run_sequential(n: int):
    return [await fake_io_call(i) for i in range(n)]

async def run_concurrent(n: int):
    tasks = [fake_io_call(i) for i in range(n)]
    return await asyncio.gather(*tasks)

n = 10
start = time.perf_counter()
sequential_results = await run_sequential(n)
sequential_time = time.perf_counter() - start

start = time.perf_counter()
concurrent_results = await run_concurrent(n)
concurrent_time = time.perf_counter() - start

print(f"sequential: {sequential_time:.3f}s, concurrent: {concurrent_time:.3f}s")
assert sequential_results == concurrent_results
assert concurrent_time < sequential_time  # concurrency wins for I/O-bound work


sequential: 0.512s, concurrent: 0.051s


🔬 **Experiment 2 — CPU-bound: multiprocessing beats threading (because of the GIL).**

⚠️ **Common Mistake:** on macOS/Windows, `multiprocessing` defaults to the `spawn` start method,
which re-imports your code in each worker process. A function defined directly in a notebook's
`__main__` can't be pickled/re-imported that way, so we write the worker function to a tiny
module file on disk first and import it — the same reason real projects put multiprocessing
worker functions in an importable module rather than a notebook cell or `__main__` script.

In [7]:
from pathlib import Path

worker_module_code = '''
def cpu_bound_work(n: int) -> int:
    total = 0
    for i in range(n):
        total += i * i
    return total
'''
Path("_mp_worker_demo.py").write_text(worker_module_code)

from _mp_worker_demo import cpu_bound_work


In [8]:
import threading, multiprocessing, time

N = 2_000_000
NUM_WORKERS = 4

# Sequential baseline
start = time.perf_counter()
for _ in range(NUM_WORKERS):
    cpu_bound_work(N)
sequential_time = time.perf_counter() - start

# Threading: GIL means this usually does NOT get faster than sequential
start = time.perf_counter()
threads = [threading.Thread(target=cpu_bound_work, args=(N,)) for _ in range(NUM_WORKERS)]
for t in threads: t.start()
for t in threads: t.join()
threading_time = time.perf_counter() - start

# Multiprocessing: separate processes, separate GILs -> real parallelism
start = time.perf_counter()
with multiprocessing.Pool(NUM_WORKERS) as pool:
    pool.map(cpu_bound_work, [N] * NUM_WORKERS)
multiprocessing_time = time.perf_counter() - start

print(f"sequential:     {sequential_time:.3f}s")
print(f"threading:      {threading_time:.3f}s  (expect ~= sequential, GIL-bound)")
print(f"multiprocessing:{multiprocessing_time:.3f}s  (expect < sequential, true parallelism)")

# Threading should not be meaningfully faster than sequential for CPU-bound work.
assert threading_time > sequential_time * 0.7

Path("_mp_worker_demo.py").unlink()  # cleanup the temporary worker module


sequential:     0.239s
threading:      0.262s  (expect ~= sequential, GIL-bound)
multiprocessing:0.132s  (expect < sequential, true parallelism)


⚠️ **Note:** exact multiprocessing speedup depends on the number of CPU cores available in
this environment and process-startup overhead for small `N`; the assertion above only checks
the GIL-bound threading behavior, which is the deterministic part of this experiment.

## 6. Connection to what's next

This same CPU-bound vs I/O-bound distinction is exactly why PyTorch's `DataLoader` uses
`num_workers` (separate **processes**, not threads) to parallelize data loading without fighting
the GIL — covered in `21_pytorch_data` on Day 3.

## 7. Interview questions
1. Why does the GIL not prevent `multiprocessing` from achieving real parallelism?
2. Why would `asyncio` beat `threading` for 1,000 concurrent network calls, in both speed and resource usage?
3. What's a `DictReader`/`DictWriter` useful for over the plain `csv.reader`/`writer`?
4. Why use `logging` instead of `print` in library code?
5. What does a named capture group in a regex (`(?P<name>...)`) give you over a plain group?

## 8. Exercises
1. Write a regex that extracts all `key=value` pairs from a log line into a dict.
2. Write an `asyncio` function that fetches "10 items" concurrently with a cap of 3 at a time (hint: `asyncio.Semaphore`).
3. Re-run the CPU-bound experiment with `NUM_WORKERS` set to your machine's core count and compare the speedup ratio.

## 9. Summary

✅ `json`/`csv`/`pathlib`/`re`/`logging` cover most data-pipeline plumbing. For concurrency:
I/O-bound → `asyncio` or `threading`; CPU-bound → `multiprocessing`, because the GIL blocks real
thread-level parallelism for Python bytecode. This closes out `01_python` — next: `02_numpy`,
where these same ideas get vectorized.